In [222]:
# basic imports

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph
from qiskit_optimization.algorithms import CplexOptimizer
# quantum imports
from qiskit_optimization.applications import Maxcut
from qiskit.circuit import Parameter,QuantumCircuit
from qiskit_optimization.translators import from_docplex_mp
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector
# Pre-defined ansatz circuit and operator class for Hamiltonian
from qiskit.circuit.library import efficient_su2
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit.circuit.library import QAOAAnsatz
# SciPy minimizer routine
from scipy.optimize import minimize
from qiskit.primitives import BackendEstimatorV2, BackendSamplerV2
from qiskit_aer import AerSimulator


In [223]:
from itertools import combinations

import numpy as np
import rustworkx as rx
from scipy.optimize import minimize

from qiskit.circuit.library import efficient_su2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Session
from rustworkx.visualization import mpl_draw
from qiskit_optimization.applications import Maxcut, Knapsack
# service = QiskitRuntimeService()
backend = AerSimulator(method="matrix_product_state")

estimator = BackendEstimatorV2(backend=backend)
sampler = BackendSamplerV2(backend=backend)


In [224]:
def calc_cut_size(graph, partition0, partition1):
    """Calculate the weighted cut size of the given partitions of the graph."""

    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        weight = data.get("weight", 1)
        if edge0 in partition0 and edge1 in partition1:
            cut_size += weight
        elif edge0 in partition1 and edge1 in partition0:
            cut_size += weight
    return cut_size


In [225]:
# Number of items
num_items = 5

# Random weights and values
np.random.seed(42)
weights = np.random.randint(1, 10, size=num_items)
values = np.random.randint(10, 50, size=num_items)

# Capacity of the knapsack
capacity = int(0.6 * np.sum(weights))

# Create the Knapsack problem
knapsack = Knapsack(values.tolist(), weights.tolist(), capacity)

# Convert the problem to a QuadraticProgram
problem = knapsack.to_quadratic_program()

converter = QuadraticProgramToQubo()
qubo = converter.convert(problem)

# classical solution
# Solve the problem using CplexOptimizer
optimizer = CplexOptimizer()
result = optimizer.solve(problem)

print("Exact Solution:")
print(result.fval)
print(result.x)


Exact Solution:
93.0
[1. 1. 0. 0. 1.]


In [226]:
# convert problem to a QUBO
converter = QuadraticProgramToQubo()
qubo = converter.convert(problem)
print(qubo.export_as_lp_string())


/var/folders/tm/6bh1bn3x6pgfgp8nvpknylq40000gn/T/ipykernel_878/1846862430.py:4: DeprecationWarning: The method ``qiskit_optimization.problems.quadratic_program.QuadraticProgram.export_as_lp_string()`` is deprecated as of Qiskit 0.7.0. It will be removed no earlier than 3 months after the release date. Use prettyprint instead.
  print(qubo.export_as_lp_string())


\ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: Knapsack

Minimize
 obj: - 33796 x_0 - 19328 x_1 - 38612 x_2 - 24140 x_3 - 33801 x_4
      - 4824 c0@int_slack@0 - 9648 c0@int_slack@1 - 19296 c0@int_slack@2
      - 38592 c0@int_slack@3 - 14472 c0@int_slack@4 + [ 13132 x_0^2
      + 15008 x_0*x_1 + 30016 x_0*x_2 + 18760 x_0*x_3 + 26264 x_0*x_4
      + 3752 x_0*c0@int_slack@0 + 7504 x_0*c0@int_slack@1
      + 15008 x_0*c0@int_slack@2 + 30016 x_0*c0@int_slack@3
      + 11256 x_0*c0@int_slack@4 + 4288 x_1^2 + 17152 x_1*x_2 + 10720 x_1*x_3
      + 15008 x_1*x_4 + 2144 x_1*c0@int_slack@0 + 4288 x_1*c0@int_slack@1
      + 8576 x_1*c0@int_slack@2 + 17152 x_1*c0@int_slack@3
      + 6432 x_1*c0@int_slack@4 + 17152 x_2^2 + 21440 x_2*x_3 + 30016 x_2*x_4
      + 4288 x_2*c0@int_slack@0 + 8576 x_2*c0@int_slack@1
      + 17152 x_2*c0@int_slack@2 + 34304 x_2*c0@int_slack@3
      + 12864 x_2*c0@int_slack@4 + 6700 x_3^2 + 18760 x_3*x_4
      + 2680 x_3*c0@int_slack@0 + 5360

In [227]:
# converting knapsack to qubo
from qubo_to_maxcut import *
from pce import *
linear = qubo.objective.linear.to_array()
quadratic = qubo.objective.quadratic.to_array()


In [228]:
weight_max_cut_qubo = QUBO(quadratic, linear)
weight_max_cut_qubo.linear_to_square()
max_cut_graph = weight_max_cut_qubo.to_maxcut()
print(max_cut_graph)


[[    0. 24332. 13872. 27832. 17380. 24322.  3484.  6968. 13936. 27872.
  10452.]
 [24332.     0.  7504. 15008.  9380. 13132.  1876.  3752.  7504. 15008.
   5628.]
 [13872.  7504.     0.  8576.  5360.  7504.  1072.  2144.  4288.  8576.
   3216.]
 [27832. 15008.  8576.     0. 10720. 15008.  2144.  4288.  8576. 17152.
   6432.]
 [17380.  9380.  5360. 10720.     0.  9380.  1340.  2680.  5360. 10720.
   4020.]
 [24322. 13132.  7504. 15008.  9380.     0.  1876.  3752.  7504. 15008.
   5628.]
 [ 3484.  1876.  1072.  2144.  1340.  1876.     0.   536.  1072.  2144.
    804.]
 [ 6968.  3752.  2144.  4288.  2680.  3752.   536.     0.  2144.  4288.
   1608.]
 [13936.  7504.  4288.  8576.  5360.  7504.  1072.  2144.     0.  8576.
   3216.]
 [27872. 15008.  8576. 17152. 10720. 15008.  2144.  4288.  8576.     0.
   6432.]
 [10452.  5628.  3216.  6432.  4020.  5628.   804.  1608.  3216.  6432.
      0.]]


In [229]:
max_cut = Maxcut(max_cut_graph)
problem_max_cut = max_cut.to_quadratic_program()
print(problem_max_cut.export_as_lp_string())


/var/folders/tm/6bh1bn3x6pgfgp8nvpknylq40000gn/T/ipykernel_878/972255532.py:3: DeprecationWarning: The method ``qiskit_optimization.problems.quadratic_program.QuadraticProgram.export_as_lp_string()`` is deprecated as of Qiskit 0.7.0. It will be removed no earlier than 3 months after the release date. Use prettyprint instead.
  print(problem_max_cut.export_as_lp_string())


\ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: Max-cut

Maximize
 obj: 170450 x_0 + 103124 x_1 + 62112 x_2 + 115736 x_3 + 76340 x_4 + 103114 x_5
      + 16348 x_6 + 32160 x_7 + 62176 x_8 + 115776 x_9 + 47436 x_10 + [
      - 97328 x_0*x_1 - 55488 x_0*x_2 - 111328 x_0*x_3 - 69520 x_0*x_4
      - 97288 x_0*x_5 - 13936 x_0*x_6 - 27872 x_0*x_7 - 55744 x_0*x_8
      - 111488 x_0*x_9 - 41808 x_0*x_10 - 30016 x_1*x_2 - 60032 x_1*x_3
      - 37520 x_1*x_4 - 52528 x_1*x_5 - 7504 x_1*x_6 - 15008 x_1*x_7
      - 30016 x_1*x_8 - 60032 x_1*x_9 - 22512 x_1*x_10 - 34304 x_2*x_3
      - 21440 x_2*x_4 - 30016 x_2*x_5 - 4288 x_2*x_6 - 8576 x_2*x_7
      - 17152 x_2*x_8 - 34304 x_2*x_9 - 12864 x_2*x_10 - 42880 x_3*x_4
      - 60032 x_3*x_5 - 8576 x_3*x_6 - 17152 x_3*x_7 - 34304 x_3*x_8
      - 68608 x_3*x_9 - 25728 x_3*x_10 - 37520 x_4*x_5 - 5360 x_4*x_6
      - 10720 x_4*x_7 - 21440 x_4*x_8 - 42880 x_4*x_9 - 16080 x_4*x_10
      - 7504 x_5*x_6 - 15008 x_5*x_7 - 30016 x_5*

In [230]:
# the graph
import networkx as nx
import matplotlib.pyplot as plt
def create_graph_from_weight_matrix(w):
    G = nx.Graph()
    n = len(w)

    # Add nodes
    for i in range(n):
        G.add_node(i)

    # Add edges with weights, ignoring zero-weight edges
    for i in range(n):
        for j in range(i + 1, n):
            if w[i, j] != 0:
                G.add_edge(i, j, weight=w[i, j])

    return G

def draw_graph(G, colors, pos):
    default_axes = plt.axes(frameon=True)
    nx.draw_networkx(G, node_color=colors, node_size=600, alpha=0.8, ax=default_axes, pos=pos)
    edge_labels = nx.get_edge_attributes(G, "weight")
    nx.draw_networkx_edge_labels(G, pos=pos, edge_labels=edge_labels)


In [231]:
graph = create_graph_from_weight_matrix(max_cut_graph)
draw_graph(graph, "lightblue", nx.spring_layout(graph))


In [232]:
from pce import PauliCorrelationEncoding, MaxCutUtility
from qubo_to_maxcut import QUBO


from pce.pauli_correlation_encoding import PauliCorrelationEncoding


pauli_encoder = PauliCorrelationEncoding()
num_nodes = problem_max_cut.get_num_binary_vars()
k = 2


num_qubits = pauli_encoder.find_n(num_nodes, k)
print(f"Number of qubits required for k={k}: {num_qubits}")


Number of qubits required for k=2: 4


In [234]:

pauli_strings = SparsePauliOp(pauli_encoder.generate_pauli_strings(num_qubits,num_nodes, k))


print(f"We can encode the problem with {num_qubits} qubits using {len(pauli_strings)} Pauli strings using k={k} compression,which are {pauli_strings}")



We can encode the problem with 4 qubits using 11 Pauli strings using k=2 compression,which are SparsePauliOp(['XXII', 'XIXI', 'XIIX', 'IXXI', 'IXIX', 'IIXX', 'YYII', 'YIYI', 'YIIY', 'IYYI', 'IYIY'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j])


In [235]:
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

print("List 1:", node_x)
print("List 2:", node_y)
print("List 3:", node_z)


List 1: [0, 1, 2]
List 2: [3, 4, 5]
List 3: [6, 7, 8, 9, 10]


In [236]:
def build_pauli_correlation_encoding(pauli, node_list, n, k=2):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]] = pauli, pauli
        pauli_correlation_encoding.append(("".join(paulis)[::-1], 1))

    hamiltonian = []
    for pauli, weight in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(pauli, weight)]))

    return hamiltonian


pauli_correlation_encoding_x = build_pauli_correlation_encoding(
    "X", node_x, num_qubits
)
pauli_correlation_encoding_y = build_pauli_correlation_encoding(
    "Y", node_y, num_qubits
)
pauli_correlation_encoding_z = build_pauli_correlation_encoding(
    "Z", node_z, num_qubits
)


In [237]:
# Build the quantum circuit
qc = efficient_su2(num_qubits, ["ry", "rz"], reps=2)


In [238]:
# Optimize the circuit

pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
qc = pm.run(qc)


In [239]:
def flatten_hamiltonian_terms(hamiltonian):
    if isinstance(hamiltonian, SparsePauliOp):
        return [op for op in hamiltonian]

    flattened = []
    for item in hamiltonian:
        if isinstance(item, list):
            flattened.extend(item)
        else:
            flattened.append(item)
    return flattened


def loss_func_estimator(x, ansatz, hamiltonian, estimator, graph):
    """
    Calculates the specified loss function for the given ansatz, Hamiltonian, and graph.

    The expectation values of each Pauli string in the Hamiltonian are first obtained
    by running the ansatz on the quantum backend. These expectation values are then
    passed through the nonlinear function tanh(alpha * prod_i). The loss function is
    subsequently computed from these transformed values.
    """
    observables = flatten_hamiltonian_terms(hamiltonian)
    job = estimator.run([(ansatz, observables, x)])
    result = job.result()[0]

    # calculate the loss function
    node_exp_map = {}
    for idx, ev in enumerate(np.atleast_1d(result.data.evs)):
        node_exp_map[idx] = ev

    loss = 0
    alpha = num_qubits
    for edge0, edge1 in graph.edges():
        loss += np.tanh(alpha * node_exp_map[edge0]) * np.tanh(
            alpha * node_exp_map[edge1]
        )

    regulation_term = 0
    for i in range(len(graph.nodes())):
        regulation_term += np.tanh(alpha * node_exp_map[i]) ** 2
    regulation_term = regulation_term / len(graph.nodes())
    regulation_term = regulation_term**2
    beta = 1 / 2
    v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4
    regulation_term = beta * v * regulation_term

    loss = loss + regulation_term

    global experiment_result
    print(f"Iter {len(experiment_result)}: {loss}")
    experiment_result.append({"loss": loss, "exp_map": node_exp_map})
    return loss


In [240]:
pce = [op.apply_layout(qc.layout) for op in pauli_strings]


In [241]:
# Run the optimization using Session

with Session(backend=backend) as session:
    estimator = Estimator(mode=session)

    experiment_result = []

    def loss_func(x):
        return loss_func_estimator(x, qc, pce, estimator, graph)

    np.random.seed(42)
    initial_params = np.random.rand(qc.num_parameters)
    result = minimize(
        loss_func, initial_params, method="COBYLA", options={"maxiter": 50}
    )
print(result)


Iter 0: 10.9985931743654
Iter 1: 2.6891103933024865
Iter 2: 0.6084334370742406
Iter 3: 0.937195358253804
Iter 4: 1.6853968782895565
Iter 5: 1.2398361265683318
Iter 6: 4.499887490046014
Iter 7: 0.43008127131593943
Iter 8: 1.2219029495351719
Iter 9: 0.757361650673837
Iter 10: 5.15771393614401
Iter 11: -0.07827786045918361
Iter 12: 0.3126629111440906
Iter 13: 0.7185257013861661
Iter 14: 0.5010472740527605
Iter 15: 3.833396621966728
Iter 16: 0.6428262284880533
Iter 17: -0.388152716568936
Iter 18: 1.2494539619157163
Iter 19: 0.20201347492194466
Iter 20: 0.11990897665412836
Iter 21: 0.9527231894803863
Iter 22: 0.17095081203405793
Iter 23: 0.9458022459771609
Iter 24: 0.23320842690518884
Iter 25: 0.8567489862802145
Iter 26: 2.7993182754435977
Iter 27: 0.8084970703474448
Iter 28: -0.3178998297685913
Iter 29: 0.07402069705053593
Iter 30: -0.19071006431495996
Iter 31: -0.009492838871531628
Iter 32: -0.3789380911599327
Iter 33: 0.40235824917105445
Iter 34: -0.23475056433056807
Iter 35: 0.858752772

In [242]:
# Calculate the partitions based on the final expectation values
# If the expectation value is positive, the node belongs to partition 0 (par0)
# Otherwise, the node belongs to partition 1 (par1)

par0, par1 = set(), set()

for i in experiment_result[-1]["exp_map"]:
    if experiment_result[-1]["exp_map"][i] >= 0:
        par0.add(i)
    else:
        par1.add(i)
print(par0, par1)


{0, 1, 3, 7, 8, 10} {2, 4, 5, 6, 9}


In [243]:
best_bits = []
cur_bits = []

for i in experiment_result[-1]["exp_map"]:
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)
print(cur_bits)


[1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1]


In [244]:
# Convert MaxCut solution to Knapsack solution
# Drop the first element (node 0 is the auxiliary node added during QUBO->MaxCut conversion)
result_initial = converter.interpret(cur_bits[1:])
initial_cost = problem.objective.evaluate(result_initial)
# check feasibility
inital_feasible = problem.get_feasibility_info(result_initial)[0]

print("Initial Knapsack score             :", initial_cost )
print("Initial Knapsack solution          :", result_initial)
print("Initial Knapsack solution feasible :", inital_feasible)


Initial Knapsack score             : 48.0
Initial Knapsack solution          : [1. 0. 1. 0. 0.]
Initial Knapsack solution feasible : True


In [246]:
# Compute initial MaxCut cost
cut_size = calc_cut_size(graph, par0, par1)
print(f"Initial MaxCut cut size: {cut_size}")

# --- Local search at MaxCut level (single-node flip, 1-opt) ---
# For each node, try flipping its partition assignment and keep the
# best improvement. Repeat until no single flip improves the cut.
best_bits = cur_bits.copy()
best_cut = cut_size
improved = True

while improved:
    improved = False
    for node in range(len(best_bits)):
        flipped_bits = best_bits.copy()
        flipped_bits[node] = 1 - flipped_bits[node]  # flip this node's partition

        cur_partition = [set(), set()]
        for i, bit in enumerate(flipped_bits):
            if bit > 0:
                cur_partition[0].add(i)
            else:
                cur_partition[1].add(i)
        new_cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
        if new_cut_size > best_cut:
            best_cut = new_cut_size
            best_bits = flipped_bits
            improved = True
            print(f"  Flipped node {node}: cut size -> {best_cut}")

print(f"Best MaxCut cut size after local search: {best_cut}")
print(f"Best bits (MaxCut level):                {best_bits}")

# Convert best MaxCut bits back to Knapsack solution
# Drop node 0 (auxiliary node from QUBO->MaxCut conversion)
qubo_bitstring = list(map(int, best_bits[1:]))
result_optimized = converter.interpret(qubo_bitstring)
optimized_cost = problem.objective.evaluate(result_optimized)
optimized_feasible = problem.get_feasibility_info(result_optimized)[0]

print(f"Optimized Knapsack score             : {optimized_cost}")
print(f"Optimized Knapsack solution          : {result_optimized}")
print(f"Optimized Knapsack solution feasible : {optimized_feasible}")


Initial MaxCut cut size: 247730.0
  Flipped node 1: cut size -> 257054.0
  Flipped node 6: cut size -> 257322.0
Best MaxCut cut size after local search: 257322.0
Best bits (MaxCut level):                [1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1]
Optimized Knapsack score             : 20.0
Optimized Knapsack solution          : [0. 0. 1. 0. 0.]
Optimized Knapsack solution feasible : True
